In [1]:
import torch
from torch.optim import Adam
from transformers import BertForMaskedLM
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score

In [2]:
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

device

'cuda'

In [3]:
import pickle

with open('./data/mimic_icd_data_dict.pkl', 'rb') as f:
    datadict = pickle.load(f)

In [4]:
class MimicIcdDataset(Dataset):
    def __init__(self, data_dict, tokenizer, tokenization_params, max_token_type_ids=100):
        """
        Setup parameters for the dataset
        Parameters
        ----------
        data_dict : dict with keys as patient ids
            training/validation data
        tokenizer : transformers tokenizer
            Tokenizer to do encoding of text
        tokenization_params : dict
            Parameters for tokenization passed to tokenizer. Note, do not set {'return_tensors': 'pt'} because this adds
            an extra dimension. Keep 'return_tensors' to default as this function does the torch tensor conversion when
            it returns the item.
            Suggested parameters: {'max_length': 512, 'truncation': True, 'padding': 'max_length'}
        """
        super().__init__()
        self.data = data_dict
        self.tokenizer = tokenizer
        self.params = tokenization_params
        self.patients = list(self.data.keys())
        self.len = len(self.data)
        self.max_token_type_ids = max_token_type_ids

    def __len__(self):
        return self.len

    def __getitem__(self, index):
        """Processing when returning an item from the Dataset. Converts text into encoding using tokenizer.
        Converts items from encoding to PyTorch tensors
        Parameters
        ----------
        index : int
            index of item to access
        Returns
        -------
        dict
            Output encoding from the tokenizer on the text.
            Relevant keys are ['input_ids', 'attention_mask', 'token_type_ids', 'label']
            What keys are present may depend on tokenizer parameters
        """
        k = self.patients[index]
        data = self.data[k]
        codes = data['icd_code']
        # Get coding embedding ids
        encoding = self.tokenizer(codes, **self.params)
        # Add token type ids for visit sequence
        num_codes_to_keep = min(self.params['max_length'] - 2, len(codes))
        encoding['token_type_ids'][1:num_codes_to_keep+1] = data['visit_order'][:num_codes_to_keep]
        encoding = {key: torch.as_tensor(val) for key, val in encoding.items()}
        return encoding


In [5]:
tokenization_params = {
    'max_length': 128,
    'truncation': True,
    'padding': 'max_length',
    'is_split_into_words': True,
    'return_special_tokens_mask': True
}

In [6]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('./data/tokenizer-icd-final')

In [7]:
dataset = MimicIcdDataset(datadict, tokenizer, tokenization_params)

In [8]:
BATCH_SIZE = 64

LEARNING_RATE = 1e-4

EPOCHS = 5

DROPOUT_PROB = 0

In [9]:
from transformers import DataCollatorForLanguageModeling
from torch.utils.data import DataLoader

dataloader = DataLoader(dataset, shuffle=True, batch_size=BATCH_SIZE,
    collate_fn=DataCollatorForLanguageModeling(tokenizer),
    num_workers=4, pin_memory=True)

In [10]:
from transformers import BertConfig

config = BertConfig(
    vocab_size=tokenizer.vocab_size,
    # hidden_size=512,
    # num_hidden_layers=6,
    # num_attention_heads=8,
    # intermediate_size=2048,
    hidden_dropout_prob=DROPOUT_PROB,
    attention_probs_dropout_prob=DROPOUT_PROB,
    max_position_embeddings=tokenization_params['max_length'],
    type_vocab_size=100,
    initializer_range=0.01
)

In [11]:
model = BertForMaskedLM(config)
model.to(device)

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(26148, 768, padding_idx=0)
      (position_embeddings): Embedding(128, 768)
      (token_type_embeddings): Embedding(100, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0): BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True

In [12]:
# from apex.optimizers import FusedAdam


In [13]:
optimizer = Adam(params=model.parameters(), lr=LEARNING_RATE)
# optimizer_fused = FusedAdam(params=model.parameters(), lr = LEARNING_RATE)

In [14]:
from transformers import get_cosine_schedule_with_warmup

lr_scheduler = get_cosine_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=6000, num_training_steps=30_000)

In [15]:
from transformers import Trainer
from transformers import TrainingArguments
import numpy as np
from datasets import load_metric

In [16]:
metric = load_metric('accuracy')

In [17]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [18]:
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        # forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        logits = logits.permute(0, 2, 1)
        # compute custom loss (suppose one has 3 labels with different weights)
        loss_fct = torch.nn.CrossEntropyLoss()
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [21]:
trainingArgs = TrainingArguments(
    output_dir='../bert_on_mimic/output',
    do_eval = True,
    evaluation_strategy="epoch",
    per_device_train_batch_size=BATCH_SIZE, 
    per_device_eval_batch_size=BATCH_SIZE,
    logging_strategy="epoch", 
    gradient_accumulation_steps=8,
    fp16 = True,
    bf16 = False,
    tf32=False,

)

For all training arguments that are possible to define: https://huggingface.co/docs/transformers/v4.21.3/en/main_classes/trainer#transformers.TrainingArguments 

For faster processing:
- Increase batch size
- Increase gradient accumulation steps
- Use fp16 or bf16 (fastest)

In [22]:
trainer = CustomTrainer(
    model,
    args = trainingArgs,
    train_dataset = dataset,
    compute_metrics=compute_metrics,
    tokenizer = tokenizer,
    optimizers=(optimizer, lr_scheduler),
    data_collator = DataCollatorForLanguageModeling(tokenizer)
)

Using cuda_amp half precision backend


In [23]:
trainer.train()

***** Running training *****
  Num examples = 190179
  Num Epochs = 3
  Instantaneous batch size per device = 64
  Total train batch size (w. parallel, distributed & accumulation) = 512
  Gradient Accumulation steps = 8
  Total optimization steps = 1113
The following columns in the training set don't have a corresponding argument in `BertForMaskedLM.forward` and have been ignored: special_tokens_mask. If special_tokens_mask are not expected by `BertForMaskedLM.forward`,  you can safely ignore this message.


Epoch,Training Loss,Validation Loss
